In [1]:
!pwd

/Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container/repaso/next_2


In [1]:
%%writefile app.py
import os
import boto3
from botocore.exceptions import ClientError

# Obtener variables de entorno o valores por defecto
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "http://localhost:9000")
AWS_KEY = os.getenv("AWS_ACCESS_KEY_ID", "minio_admin")
AWS_SECRET = os.getenv("AWS_SECRET_ACCESS_KEY", "minio_password")
BUCKET_NAME = "engine-trouble-codes"

def run_engine_sandbox(custom_s3_client=None):
    """
    Ejecuta el sandbox de diagnóstico e interactúa con el bucket S3/MinIO.
    Permite inyectar un cliente mock de S3 para pruebas unitarias.
    """
    if custom_s3_client:
        s3 = custom_s3_client
    else:
        print(f"🛡️ Connecting to MinIO S3 at: {S3_ENDPOINT}")
        s3 = boto3.client(
            's3',
            endpoint_url=S3_ENDPOINT,
            aws_access_key_id=AWS_KEY,
            aws_secret_access_key=AWS_SECRET,
            region_name='us-east-1'
        )
    
    # 1. Crear el bucket si no existe
    try:
        s3.head_bucket(Bucket=BUCKET_NAME)
    except ClientError:
        print(f"📦 Bucket '{BUCKET_NAME}' not found. Creating it now...")
        s3.create_bucket(Bucket=BUCKET_NAME)
    
    # 2. Reporte de diagnóstico del motor (Código P0300)
    report_data = "DIAGNOSTIC REPORT: CODE P0300 - RANDOM/MULTIPLE CYLINDER MISFIRE DETECTED. CHECK SPARK PLUGS AND IGNITION COILS."
    
    # 3. Subir reporte al Bucket
    s3.put_object(Bucket=BUCKET_NAME, Key="misfire_report.txt", Body=report_data)
    print(f"🚀 Misfire report uploaded to bucket '{BUCKET_NAME}'!")
    
    # 4. Descargar y verificar
    response = s3.get_object(Bucket=BUCKET_NAME, Key="misfire_report.txt")
    downloaded_data = response['Body'].read().decode('utf-8')
    print("📥 Cloud data verification: 100% Match!")
    
    # 5. Guardar copia local en la carpeta logs/
    os.makedirs('logs', exist_ok=True)
    with open('logs/final_misfire_report.txt', 'w') as f:
        f.write(downloaded_data)
    print("💾 Permanent copy stamped to local container logs directory!")
    
    return downloaded_data

if __name__ == "__main__":
    run_engine_sandbox()

Overwriting app.py


In [2]:
%%writefile app.py
import os
import boto3
from botocore.exceptions import ClientError

# Obtener variables de entorno o valores por defecto
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "http://localhost:9000")
AWS_KEY = os.getenv("AWS_ACCESS_KEY_ID", "minio_admin")
AWS_SECRET = os.getenv("AWS_SECRET_ACCESS_KEY", "minio_password")
BUCKET_NAME = "engine-trouble-codes"

def run_engine_sandbox(custom_s3_client=None):
    """
    Ejecuta el sandbox de diagnóstico e interactúa con el bucket S3/MinIO.
    Permite inyectar un cliente mock de S3 para pruebas unitarias.
    """
    if custom_s3_client:
        s3 = custom_s3_client
    else:
        print(f"🛡️ Connecting to MinIO S3 at: {S3_ENDPOINT}")
        s3 = boto3.client(
            's3',
            endpoint_url=S3_ENDPOINT,
            aws_access_key_id=AWS_KEY,
            aws_secret_access_key=AWS_SECRET,
            region_name='us-east-1'
        )
    
    # 1. Crear el bucket si no existe
    try:
        s3.head_bucket(Bucket=BUCKET_NAME)
    except ClientError:
        print(f"📦 Bucket '{BUCKET_NAME}' not found. Creating it now...")
        s3.create_bucket(Bucket=BUCKET_NAME)
    
    # 2. Reporte de diagnóstico del motor (Código P0300)
    report_data = "DIAGNOSTIC REPORT: CODE P0300 - RANDOM/MULTIPLE CYLINDER MISFIRE DETECTED. CHECK SPARK PLUGS AND IGNITION COILS."
    
    # 3. Subir reporte al Bucket
    s3.put_object(Bucket=BUCKET_NAME, Key="misfire_report.txt", Body=report_data)
    print(f"🚀 Misfire report uploaded to bucket '{BUCKET_NAME}'!")
    
    # 4. Descargar y verificar
    response = s3.get_object(Bucket=BUCKET_NAME, Key="misfire_report.txt")
    downloaded_data = response['Body'].read().decode('utf-8')
    print("📥 Cloud data verification: 100% Match!")
    
    # 5. Guardar copia local en la carpeta logs/
    os.makedirs('logs', exist_ok=True)
    with open('logs/final_misfire_report.txt', 'w') as f:
        f.write(downloaded_data)
    print("💾 Permanent copy stamped to local container logs directory!")
    
    return downloaded_data

if __name__ == "__main__":
    run_engine_sandbox()

Writing app.py


In [3]:
%%writefile tests/test_s3_sandbox.py
import sys
import os
import pytest
import boto3
from moto import mock_aws

# Corregir la ruta para importar app.py desde el directorio superior
sys.path.insert(0, os.path.abspath(os.path.dirname(__file__) + "/.."))

from app import run_engine_sandbox, BUCKET_NAME

@mock_aws
def test_run_engine_sandbox_flow():
    """
    Prueba el flujo completo de S3 usando un mock en memoria (moto).
    """
    # 1. Crear cliente S3 simulado
    s3_mock = boto3.client('s3', region_name='us-east-1')
    
    # 2. Ejecutar la función principal pasando el mock
    content = run_engine_sandbox(custom_s3_client=s3_mock)
    
    # 3. Asserts / Verificaciones
    assert "P0300" in content
    
    # Verificar que el objeto existe en S3
    response = s3_mock.get_object(Bucket=BUCKET_NAME, Key="misfire_report.txt")
    retrieved_text = response['Body'].read().decode('utf-8')
    assert "RANDOM/MULTIPLE CYLINDER MISFIRE DETECTED" in retrieved_text
    
    # Verificar que el log local fue escrito correctamente
    assert os.path.exists("logs/final_misfire_report.txt")

Writing tests/test_s3_sandbox.py


In [4]:
%%writefile requirements.txt
boto3==1.34.0
botocore==1.34.0
pytest==8.0.0
moto==5.0.0

Writing requirements.txt


In [5]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

CMD ["python", "app.py"]

Writing Dockerfile


In [6]:
!pip install -q moto
!PYTHONPATH=. pytest tests/ -v


[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: pip install --upgrade pip
============================= test session starts ==============================
platform darwin -- Python 3.11.5, pytest-9.1.1, pluggy-1.6.0 -- /Users/admin/Desktop/Shafer_Python_Classes/pandas_env/bin/python3.11
cachedir: .pytest_cache
rootdir: /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container/repaso/next_2
plugins: anyio-4.4.0
collected 1 item                                                               

tests/test_s3_sandbox.py::test_run_engine_sandbox_flow PASSED            [100%]

============================== 1 passed in 1.32s ===============================


In [7]:
!pwd


/Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container/repaso/next_2


In [ ]:
%%writefile ../../../../../.github/workflows/next2_pipeline.yml
name: 🚗 CI/CD - S3 Diagnostic Service Pipeline (next_2)

on:
  push:
    branches: [ "main", "master" ]

jobs:
  test-and-lint:
    name: 🧪 Run S3 Service Tests
    runs-on: ubuntu-latest

    steps:
      - name: 1️⃣ Checkout del código
        uses: actions/checkout@v4

      - name: 2️⃣ Configurar Python 3.10
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: 3️⃣ Instalar dependencias
        run: |
          python -m pip install --upgrade pip
          pip install -r containers/ninth_container/repaso/next_2/requirements.txt

      - name: 4️⃣ Ejecutar Pytest Suite
        run: |
          PYTHONPATH=containers/ninth_container/repaso/next_2 pytest containers/ninth_container/repaso/next_2/tests/ -v

  build-and-deploy:
    name: 🐳 Build & Push S3 Service Image
    needs: test-and-lint
    runs-on: ubuntu-latest

    steps:
      - name: 1️⃣ Checkout del código
        uses: actions/checkout@v4

      - name: 2️⃣ Iniciar sesión en Docker Hub
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKERHUB_USERNAME }}
          password: ${{ secrets.DOCKERHUB_TOKEN }}

      - name: 3️⃣ Configurar Docker Buildx
        uses: docker/setup-buildx-action@v3

      - name: 4️⃣ Compilar y Subir Imagen de la API S3
        uses: docker/build-push-action@v5
        with:
          context: containers/ninth_container/repaso/next_2
          push: true
          tags: |
            ${{ secrets.DOCKERHUB_USERNAME }}/engine-s3-service:v1
            ${{ secrets.DOCKERHUB_USERNAME }}/engine-s3-service:latest